# Ditto — CIFAR-10, 30 clients, 50 rounds

**Ditto** (Li, Hu, Beirami, Smith — ICML 2021) on CIFAR-10, matching the
exact experimental conditions of the Hierarchical Ensemble fixed-spec run
for a fair, direct comparison:

- **Dataset:** CIFAR-10 (full 50k training / 10k test)
- **30 clients**, Dirichlet `alpha=0.1` (strongly Non-IID)
- **50 communication rounds**
- **Model:** ResNet9 (same architecture as the Hierarchical Ensemble run)
- **`local_lr=0.01`, `local_steps=2`** — identical to the Hierarchical Ensemble config
- **Seed:** 42 (fully reproducible with the seeding fix in the repo)

### What Ditto does

Each client maintains a **personalized model `v_k`** alongside the shared
global model `w*`. The personalized model is updated via a regularized loss
that pulls it toward the current global model (Algorithm 1 in the paper):

```
v_k ← v_k - η * (∇F_k(v_k) + λ*(v_k - w_t))
```

where `w_t` is the global model at the start of the round (snapshotted and
held fixed for the duration of that round's local steps). The global model
is updated by ordinary local SGD and aggregated via FedAvg — exactly the
same as the Hierarchical Ensemble's Root model.

`DITTO_LAMBDA = 1.0` is used here (one of the paper's own candidate values
for non-attack scenarios). Both models (global and personalized) are
reported each round.

### How to use this notebook
1. Run the cells top to bottom.
2. **Section 2** asks you to upload `Topology-aware-FDL.zip`.
3. **Section 4** runs Ditto for 50 rounds with per-round checkpointing —
   if your machine restarts, just re-run this cell and it will resume from
   the last completed round automatically.
4. **Section 5** downloads all results.

> **Tip:** Use a GPU runtime — 50 rounds of CIFAR-10 with ResNet9 across
> 30 clients is heavy on CPU.


In [ ]:
# @title 1. Check GPU and basic environment
import subprocess, sys

gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                            capture_output=True, text=True)
if gpu_check.returncode == 0:
    print("GPU detected:", gpu_check.stdout.strip())
else:
    print("No GPU detected. Go to Runtime > Change runtime type > GPU for faster training.")
    print("The notebook will still run on CPU, just slower.")

print("\nPython:", sys.version)


## 2. Get the code onto the machine

This notebook was originally written for Colab but also runs fine on
Paperspace, a local Jupyter server, or anything else — `IN_COLAB` below
detects which one you're on and the rest of this section adapts automatically.

**If you're on Colab:** the cell below opens a file-picker — just select
`Topology-aware-FDL.zip` (the zip provided alongside this notebook).

**If you're on Paperspace / a local Jupyter server / anything else:** Colab's
picker (`google.colab.files.upload()`) doesn't exist outside Colab. The cell
below instead shows an **upload button right in the notebook output** (via
`ipywidgets`) — this sidesteps Jupyter's separate file-browser entirely, so
there's no guessing about which folder it landed in: whatever you select gets
written straight to this kernel's working directory.

If the widget doesn't render for some reason (older Jupyter frontend,
`ipywidgets` blocked, etc.), 2a-iii below is a no-widget fallback: upload the
zip via Jupyter's file browser (upload-arrow icon, left panel) into the
**same folder this notebook file is in**, then run 2a-iii and set
`PROJECT_ZIP_PATH` if it isn't found automatically.


In [ ]:
# @title 2a-i. Detect environment
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Environment: {'Colab' if IN_COLAB else 'Non-Colab Jupyter (e.g. Paperspace)'}")

if IN_COLAB:
    WORK_DIR = "/content"
else:
    # os.getcwd() raises FileNotFoundError if the kernel's current directory
    # was deleted out from under it (e.g. after rm -rf'ing a folder while the
    # kernel was sitting inside it) -- a plain kernel restart doesn't always
    # clear this on every Jupyter host. Fall back to a directory that's
    # guaranteed to exist instead of crashing here.
    try:
        WORK_DIR = os.getcwd()
    except FileNotFoundError:
        for _candidate in ("/notebooks", os.path.expanduser("~"), "/tmp"):
            if os.path.isdir(_candidate):
                WORK_DIR = _candidate
                break
        else:
            WORK_DIR = "/tmp"
        os.chdir(WORK_DIR)
        print(f"NOTE: the kernel's previous working directory no longer exists on disk "
              f"(this usually means a folder it was sitting inside got deleted). "
              f"Falling back to {WORK_DIR} -- if this isn't where you want to work, "
              f"restart the kernel from Jupyter's menu (not just re-running cells) "
              f"and try again before continuing.")

PROJECT_DIR = os.path.join(WORK_DIR, "Topology-aware-FDL-main")
print(f"Files will be set up under: {WORK_DIR}")


In [ ]:
# @title 2a-ii. Upload the zip (Colab file-picker, or an in-notebook button elsewhere)
zip_path = None

if os.path.exists(PROJECT_DIR):
    print(f"Project already present at {PROJECT_DIR} -- skip to 2a-iv (or delete that folder to re-upload).")
elif IN_COLAB:
    from google.colab import files
    print("Please select Topology-aware-FDL.zip...")
    uploaded = files.upload()
    zip_path = os.path.join(os.getcwd(), next(iter(uploaded.keys())))
else:
    # No Colab picker here -- render an upload button directly in this cell's
    # output instead of relying on Jupyter's separate file browser (which is
    # what caused the FileNotFoundError: there's no reliable single path
    # every Jupyter frontend/host uses for browser-panel uploads, but a
    # widget rendered by the kernel itself always writes to a path the
    # kernel controls).
    try:
        import ipywidgets as widgets
        from IPython.display import display

        UPLOAD_TARGET = os.path.join(WORK_DIR, "Topology-aware-FDL.zip")
        upload_widget = widgets.FileUpload(accept=".zip", multiple=False, description="Select zip")
        save_button = widgets.Button(description="Save upload", button_style="success")
        status_label = widgets.Label(value="1) Click 'Select zip' and choose Topology-aware-FDL.zip  2) Click 'Save upload'")

        def _on_save_clicked(b):
            global zip_path
            if not upload_widget.value:
                status_label.value = "No file selected yet -- click 'Select zip' first, then 'Save upload'."
                return
            # ipywidgets 8.x: .value is a tuple of dicts, each with a 'content' key
            # (memoryview/bytes). Verified directly against the installed widget
            # source rather than assumed -- older ipywidgets 7.x used a different
            # dict-keyed-by-filename shape, but pip installs 8.x by default now.
            uploaded_file = upload_widget.value[0]
            with open(UPLOAD_TARGET, "wb") as f:
                f.write(bytes(uploaded_file["content"]))
            zip_path = UPLOAD_TARGET
            status_label.value = f"Saved to {UPLOAD_TARGET} ({os.path.getsize(UPLOAD_TARGET)} bytes). Now run 2a-iv."

        save_button.on_click(_on_save_clicked)
        display(widgets.VBox([upload_widget, save_button, status_label]))
    except ImportError:
        print("ipywidgets isn't available -- use the no-widget fallback in 2a-iii instead:")
        print("upload the zip via Jupyter's file browser, then set PROJECT_ZIP_PATH there.")


In [ ]:
# @title 2a-iii. No-widget fallback (only needed if 2a-ii's widget didn't render)
# If you uploaded via Jupyter's own file browser instead of the widget above,
# set the full path here and run this cell -- skip it if 2a-ii already worked.

PROJECT_ZIP_PATH = None  # e.g. "/notebooks/Topology-aware-FDL.zip" -- check Jupyter's file browser for the real path

if not os.path.exists(PROJECT_DIR) and zip_path is None:
    if PROJECT_ZIP_PATH:
        zip_path = PROJECT_ZIP_PATH
    else:
        import glob
        candidates = (
            glob.glob(os.path.join(WORK_DIR, "Topology-aware-FDL*.zip"))
            + glob.glob(os.path.join(os.getcwd(), "Topology-aware-FDL*.zip"))
            + glob.glob(os.path.join(os.path.expanduser("~"), "Topology-aware-FDL*.zip"))
            + glob.glob("/notebooks/Topology-aware-FDL*.zip")
        )
        if candidates:
            zip_path = candidates[0]
            print(f"Found zip at: {zip_path}")
        else:
            print("Still not found automatically. Set PROJECT_ZIP_PATH above to the exact path shown")
            print("in Jupyter's file browser for the file you uploaded, then re-run this cell.")
else:
    print("Already have a zip path or project dir from 2a-ii -- nothing to do here.")


In [ ]:
# @title 2a-iv. Extract and set up the project (run after 2a-ii or 2a-iii succeeded)
import zipfile, shutil

if os.path.exists(PROJECT_DIR):
    print(f"Project already present at {PROJECT_DIR}, skipping extract.")
else:
    assert zip_path and os.path.exists(zip_path), (
        f"No zip found at '{zip_path}' -- finish 2a-ii (widget) or 2a-iii (fallback) first."
    )

    extract_dir = os.path.join(WORK_DIR, "_extracted")
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)
    os.makedirs(extract_dir)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    # Handle both "zip contains the project folder itself" and
    # "zip contains the project files at top level" layouts.
    entries = [e for e in os.listdir(extract_dir) if not e.startswith("__MACOSX")]
    if len(entries) == 1 and os.path.isdir(os.path.join(extract_dir, entries[0])):
        shutil.move(os.path.join(extract_dir, entries[0]), PROJECT_DIR)
    else:
        shutil.move(extract_dir, PROJECT_DIR)

    print(f"Project ready at {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# @title 2b. Alternative — Google Drive (Colab only; skip if you used 2a)
# Uncomment and run if you're on Colab and have already placed the unzipped
# project folder in your Drive. Not applicable on Paperspace/other Jupyter.

# from google.colab import drive
# drive.mount('/content/drive')
# import os
# PROJECT_DIR = "/content/drive/MyDrive/Topology-aware-FDL-main"  # adjust path as needed
# os.chdir(PROJECT_DIR)
# print("Working directory:", os.getcwd())


## 3. Install dependencies

In [ ]:
# @title 3. Install dependencies
# Colab already ships with torch, numpy, pandas, matplotlib, seaborn, scikit-learn.
# This installs anything missing (networkx, pyyaml, tqdm) without touching
# Colab's preinstalled torch/CUDA build.
#
# pydantic is handled separately and deliberately upgraded: this codebase uses
# pydantic v2-only APIs (model_validator, model_dump). A plain
# "pip install pydantic" does nothing if some pydantic is already present --
# which is exactly what happens on environments that ship an old v1 by default
# (e.g. Paperspace's base image) -- so it's pinned and force-upgraded here
# instead of left to chance.
#
# Some hosts (Paperspace's included) mark the system Python as "externally
# managed" (PEP 668) and refuse a plain pip install. --break-system-packages
# overrides that. This is fine in a disposable cloud notebook container; it
# would be worth avoiding on a machine you otherwise rely on.

import subprocess

def pip_install(*args):
    cmd = ["pip", "install", "-q"] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 and "externally-managed-environment" in result.stderr:
        cmd = cmd + ["--break-system-packages"]
        result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"pip install failed for: {args}")

pip_install("networkx", "pyyaml", "tqdm")
pip_install("--upgrade", "pydantic>=2.0")

import pydantic
assert pydantic.VERSION.startswith(tuple(str(v) for v in range(2, 10))), (
    f"pydantic {pydantic.VERSION} is still v1 -- this codebase needs v2+. "
    "Try restarting the kernel after this cell (some environments cache the "
    "old import in already-running processes) and re-run from Section 1."
)
print("pydantic:", pydantic.VERSION)

import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

# main.py refuses to run outside an activated virtualenv (a local-dev safety check).
# Colab notebooks never run inside one, so this sets the env var main.py checks for
# rather than editing the repo's source. Safe to ignore/remove if you've modified
# main.py's venv guard yourself.
import os
os.environ["VIRTUAL_ENV"] = "/content/colab_venv_shim"


## 4. Run Ditto

The shared setup cell below loads CIFAR-10, partitions it across 30 clients
with the same Dirichlet `alpha=0.1` split used in the Hierarchical Ensemble
run, and defines the helper functions used by the training loop.

The training cell checkpoints after every single round — both the global
model weights and all 30 clients' personalized model states are saved to
`./outputs/ditto_checkpoint.pt` after each round completes, and
`./outputs/ditto_metrics.json` is updated at the same time. If your machine
restarts or the kernel dies, just re-run the training cell and it will
resume from where it left off automatically.


In [ ]:
# @title 4a. Shared setup for Ditto and APFL
# Assumes Section 2 already ran (so we're chdir'd into the project root) and
# Section 3 already ran (torch etc. available). Re-imports here are cheap
# and make this section runnable on its own if you skip straight to it.
import os, sys, copy, json
sys.path.insert(0, ".")

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.core.model import ResNet9
from src.data.dataset import get_cifar10, partition_data, ClientDataset
from src.utils.random import set_seed  # already torch-seeding, per the Section 11 fix

# Same specification as Section 12, repeated explicitly here so this section
# doesn't silently depend on FIXED_* variables from a cell that may not have
# run in this session.
COMPARISON_SEED = 42
COMPARISON_NUM_CLIENTS = 30
COMPARISON_NON_IID_ALPHA = 0.1
COMPARISON_NUM_ROUNDS = 50
COMPARISON_LOCAL_LR = 0.01
COMPARISON_LOCAL_STEPS = 2
COMPARISON_BATCH_SIZE = 32
COMPARISON_TRAIN_SUBSET = None   # None = full 50k, matching Section 12's spec
COMPARISON_TEST_SUBSET = None
COMPARISON_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

COMPARISON_OUTPUT_DIR = "./outputs"
os.makedirs(COMPARISON_OUTPUT_DIR, exist_ok=True)

print(f"Device: {COMPARISON_DEVICE}")
set_seed(COMPARISON_SEED)

print("Loading CIFAR-10...")
comparison_train_dataset, comparison_test_dataset = get_cifar10(
    data_dir="./data",
    train_subset=COMPARISON_TRAIN_SUBSET,
    test_subset=COMPARISON_TEST_SUBSET,
)

comparison_train_indices = partition_data(
    comparison_train_dataset, COMPARISON_NUM_CLIENTS, non_iid=True,
    alpha=COMPARISON_NON_IID_ALPHA, seed=COMPARISON_SEED,
)
comparison_test_indices = partition_data(
    comparison_test_dataset, COMPARISON_NUM_CLIENTS, non_iid=True,
    alpha=COMPARISON_NON_IID_ALPHA, seed=COMPARISON_SEED,
)
comparison_client_train_sets = {
    cid: ClientDataset(comparison_train_dataset, idx) for cid, idx in comparison_train_indices.items()
}
comparison_client_test_sets = {
    cid: ClientDataset(comparison_test_dataset, idx) for cid, idx in comparison_test_indices.items()
}

_sizes = [len(v) for v in comparison_client_train_sets.values()]
print(f"Client train set sizes: min={min(_sizes)}, max={max(_sizes)}, "
      f"zero-sample clients={sum(1 for s in _sizes if s == 0)}")
assert min(_sizes) > 0, (
    "A client has zero training samples at this seed/alpha/client-count -- this would crash "
    "training. (alpha=0.1 at 30 clients was verified safe for seed=42 in earlier testing; if "
    "you change COMPARISON_SEED or COMPARISON_NUM_CLIENTS, re-verify this doesn't happen.)"
)

comparison_criterion = nn.CrossEntropyLoss()


def comparison_evaluate(model, dataset, device, criterion):
    model.eval()
    if len(dataset) == 0:
        return 0.0, 0.0
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    correct, total, total_loss = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    if total == 0:
        return 0.0, 0.0
    return 100.0 * correct / total, total_loss / total


def comparison_fedavg_aggregate(state_dicts, weights=None):
    if weights is None:
        weights = [1.0 / len(state_dicts)] * len(state_dicts)
    else:
        total = sum(weights)
        weights = [w / total for w in weights]
    avg_state = copy.deepcopy(state_dicts[0])
    for key in avg_state.keys():
        if avg_state[key].dtype in (torch.float32, torch.float64, torch.float16):
            avg_state[key] = sum(w * sd[key].float() for w, sd in zip(weights, state_dicts))
        else:
            avg_state[key] = state_dicts[0][key]
    return avg_state


print("Setup complete.")


### 4b. Ditto

`v_k <- v_k - eta*(grad F_k(v_k) + lambda*(v_k - w_t))` per Algorithm 1.
`w_t` is snapshotted at the start of each round and held fixed through that
round's local steps, matching the paper -- not the in-progress global copy,
which drifts during local SGD.

`DITTO_LAMBDA` isn't from the paper directly (it tunes this per-device from
validation data, out of scope here) -- 1.0 is one of the paper's own
candidate values for non-attack scenarios (Section 4, "Setup"), used here as
a single fixed value for a clean-data comparison.


In [ ]:
# @title 4c. Run Ditto (50 rounds -- checkpoints every round, resumable)
# If your machine might restart mid-run (common on Paperspace free/spot
# instances), this cell saves progress to disk after EVERY round -- both the
# metrics history and the actual model weights -- and automatically resumes
# from the last completed round if you just re-run this same cell after a
# restart. Nothing before the saved checkpoint is re-computed.
DITTO_LAMBDA = 1.0
DITTO_CHECKPOINT_PATH = os.path.join(COMPARISON_OUTPUT_DIR, "ditto_checkpoint.pt")
DITTO_METRICS_PATH = os.path.join(COMPARISON_OUTPUT_DIR, "ditto_metrics.json")

if os.path.exists(DITTO_CHECKPOINT_PATH):
    print(f"Found existing checkpoint at {DITTO_CHECKPOINT_PATH} -- resuming.")
    checkpoint = torch.load(DITTO_CHECKPOINT_PATH, map_location=COMPARISON_DEVICE)
    ditto_global_model = ResNet9(in_channels=3).to(COMPARISON_DEVICE)
    ditto_global_model.load_state_dict(checkpoint["global_state"])
    ditto_personalized_states = checkpoint["personalized_states"]
    ditto_history = checkpoint["history"]
    start_round = checkpoint["last_completed_round"] + 1
    print(f"Resuming from round {start_round} (rounds 1-{checkpoint['last_completed_round']} already done).")
else:
    print("No checkpoint found -- starting fresh.")
    ditto_global_model = ResNet9(in_channels=3).to(COMPARISON_DEVICE)
    print(f"ResNet9 instantiated with {sum(p.numel() for p in ditto_global_model.parameters())} parameters.")
    ditto_personalized_states = {
        cid: copy.deepcopy(ditto_global_model.state_dict()) for cid in range(COMPARISON_NUM_CLIENTS)
    }
    ditto_history = []
    start_round = 1

if start_round > COMPARISON_NUM_ROUNDS:
    print(f"Checkpoint already covers all {COMPARISON_NUM_ROUNDS} rounds -- nothing to do. "
          f"Delete {DITTO_CHECKPOINT_PATH} to force a fresh run.")

for round_num in range(start_round, COMPARISON_NUM_ROUNDS + 1):
    client_global_updates, client_sample_counts = [], []
    global_state_at_round_start = copy.deepcopy(ditto_global_model.state_dict())

    for cid in range(COMPARISON_NUM_CLIENTS):
        train_ds = comparison_client_train_sets[cid]
        if len(train_ds) == 0:
            client_global_updates.append(copy.deepcopy(global_state_at_round_start))
            client_sample_counts.append(0)
            continue

        w_k = ResNet9(in_channels=3).to(COMPARISON_DEVICE)
        w_k.load_state_dict(global_state_at_round_start)
        w_k.train()
        w_optimizer = torch.optim.SGD(w_k.parameters(), lr=COMPARISON_LOCAL_LR)

        w_t_model = ResNet9(in_channels=3).to(COMPARISON_DEVICE)
        w_t_model.load_state_dict(global_state_at_round_start)
        w_t_params = list(w_t_model.parameters())

        v_k = ResNet9(in_channels=3).to(COMPARISON_DEVICE)
        v_k.load_state_dict(ditto_personalized_states[cid])
        v_k.train()

        batch_size = min(COMPARISON_BATCH_SIZE, len(train_ds))
        loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

        for _ in range(COMPARISON_LOCAL_STEPS):
            for images, labels in loader:
                images, labels = images.to(COMPARISON_DEVICE), labels.to(COMPARISON_DEVICE)

                w_optimizer.zero_grad()
                loss_w = comparison_criterion(w_k(images), labels)
                loss_w.backward()
                w_optimizer.step()

                logits_v = v_k(images)
                loss_v = comparison_criterion(logits_v, labels)
                grads_v = torch.autograd.grad(loss_v, list(v_k.parameters()))
                with torch.no_grad():
                    for p_v, g_v, p_wt in zip(v_k.parameters(), grads_v, w_t_params):
                        p_v -= COMPARISON_LOCAL_LR * (g_v + DITTO_LAMBDA * (p_v - p_wt))

        client_global_updates.append(copy.deepcopy(w_k.state_dict()))
        client_sample_counts.append(len(train_ds))
        ditto_personalized_states[cid] = copy.deepcopy(v_k.state_dict())

    non_empty = [(s, c) for s, c in zip(client_global_updates, client_sample_counts) if c > 0]
    if non_empty:
        states, counts = zip(*non_empty)
        ditto_global_model.load_state_dict(comparison_fedavg_aggregate(list(states), weights=list(counts)))

    global_acc, global_loss = comparison_evaluate(ditto_global_model, comparison_test_dataset, COMPARISON_DEVICE, comparison_criterion)

    personalized_accs = []
    eval_model = ResNet9(in_channels=3).to(COMPARISON_DEVICE)
    for cid in range(COMPARISON_NUM_CLIENTS):
        test_ds = comparison_client_test_sets[cid]
        if len(test_ds) == 0:
            continue
        eval_model.load_state_dict(ditto_personalized_states[cid])
        acc, _ = comparison_evaluate(eval_model, test_ds, COMPARISON_DEVICE, comparison_criterion)
        personalized_accs.append(acc)
    avg_personalized_acc = float(np.mean(personalized_accs)) if personalized_accs else 0.0

    ditto_history.append({
        "round": round_num, "global_test_accuracy": global_acc, "global_test_loss": global_loss,
        "personalized_avg_accuracy": avg_personalized_acc,
    })
    print(f"Round {round_num:3d}/{COMPARISON_NUM_ROUNDS} | Global: {global_acc:6.2f}% (loss {global_loss:.3f}) | "
          f"Ditto personalized (avg): {avg_personalized_acc:6.2f}%")

    # Checkpoint after every round -- this is the part that protects you from
    # a machine restart. Written atomically-ish (write to a temp name, then
    # rename) so a crash mid-save can't leave a half-written, corrupt checkpoint.
    tmp_path = DITTO_CHECKPOINT_PATH + ".tmp"
    torch.save({
        "global_state": ditto_global_model.state_dict(),
        "personalized_states": ditto_personalized_states,
        "history": ditto_history,
        "last_completed_round": round_num,
    }, tmp_path)
    os.replace(tmp_path, DITTO_CHECKPOINT_PATH)
    with open(DITTO_METRICS_PATH, "w") as f:
        json.dump(ditto_history, f, indent=2)

print("\nDitto run complete.")


## 5. Results


In [ ]:
# @title 5a. Print final results and plot
import pandas as pd
import matplotlib.pyplot as plt
import os

metrics_path = "./outputs/ditto_metrics.json"
assert os.path.exists(metrics_path), "No results found yet -- run Section 4 first."

df = pd.DataFrame(__import__('json').load(open(metrics_path)))
display(df)

final = df.iloc[-1]
print(f"\n{'='*55}")
print("FINAL RESULTS (Ditto, round", int(final['round']), "of 50)")
print(f"{'='*55}")
print(f"Global accuracy:            {final['global_test_accuracy']:.2f}%")
print(f"Ditto personalized avg:     {final['personalized_avg_accuracy']:.2f}%")
print(f"Gain over global:           {final['personalized_avg_accuracy'] - final['global_test_accuracy']:+.2f} pts")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(df["round"], df["global_test_accuracy"], marker="o", label="Global (w*)")
axes[0].plot(df["round"], df["personalized_avg_accuracy"], marker="s", linestyle="--", label="Ditto personalized (avg)")
axes[0].set_xlabel("Round"); axes[0].set_ylabel("Test accuracy (%)"); axes[0].legend(); axes[0].grid(True)
axes[0].set_title("Ditto: accuracy over 50 rounds\nCIFAR-10, 30 clients, alpha=0.1")
axes[1].plot(df["round"], df["global_test_loss"], marker="o", label="Global (w*)")
axes[1].set_xlabel("Round"); axes[1].set_ylabel("Test loss"); axes[1].legend(); axes[1].grid(True)
axes[1].set_title("Global model loss (check for instability)")
plt.tight_layout()
plt.savefig("./outputs/ditto_results.png", dpi=120)
plt.show()


## 6. Download results

**On Colab** this triggers a browser download automatically.
**On Paperspace** the zip is written to your working directory — right-click
it in the Jupyter file browser and choose Download.


In [ ]:
# @title 6. Download all outputs
import glob, os, shutil

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

assert os.path.exists("./outputs"), "No outputs found yet."

zip_dir = "/content" if IN_COLAB else os.getcwd()
zip_path = os.path.join(zip_dir, "run_results.zip")
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive(os.path.join(zip_dir, "run_results"), "zip", "./outputs")

print(f"Zipped: ./outputs -> {zip_path}")

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
else:
    print("Not on Colab -- no auto-download available. Find the zip in the Jupyter")
    print(f"file browser at: {zip_path}")
    print("Right-click it there and choose Download, or copy it off the machine yourself.")
